In [1]:
# View and modify the working path
import os
from google.colab import drive

# View current working directory
print("Current Working Directory:", os.getcwd())

# Mount Google Drive
drive.mount('/content/gdrive')

# Change working directory to your file position
path = "/content/gdrive/My Drive/BD4H/data"
os.chdir(path)

# Confirm the change
print("Working Directory:", os.getcwd())

Current Working Directory: /content
Mounted at /content/gdrive
Working Directory: /content/gdrive/My Drive/BD4H/data


In [2]:
import pandas as pd

#Read the DIAGNOSES_ICD table and select the admission IDs related to heart failure
ICD_info = pd.read_csv('DIAGNOSES_ICD.csv.gz', usecols=['SUBJECT_ID', 'HADM_ID', 'ICD9_CODE'], compression='gzip')
heart_failure_ICD=['39891', '40201', '40211', '40291', '40401', '40403',\
                   '40411', '40413', '40491', '40493', '4280', '4281',\
                   '42820', '42821', '42822', '42823', '42830', '42831',\
                   '42832', '42833','42840', '42841', '42842', '42843', '4289']
heart_failure_events = ICD_info[ICD_info['ICD9_CODE'].isin(heart_failure_ICD)]
#Unique admission IDs related to heart failure
heart_failure_HADM = heart_failure_events['HADM_ID'].unique()
display(len(heart_failure_HADM))

#Read the ADMISSIONS table and determine the admission and discharge time of the heart failure events
admission_info = pd.read_csv('ADMISSIONS.csv', usecols=['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME'],\
                             parse_dates=['ADMITTIME', 'DISCHTIME'])
heart_failure_admittime = admission_info[admission_info['HADM_ID'].isin(heart_failure_HADM)]
#Sort the table by SUBJECT_ID (patient) and admission time
sorted_df=heart_failure_admittime.sort_values(by=['SUBJECT_ID', 'ADMITTIME'])
#Group by SUBJECT_ID and isolate all but the most recent HADM_ID for each group
most_recent_adm = sorted_df.groupby('SUBJECT_ID')['HADM_ID'].tail(1).reset_index(drop=True)
#The remaining HADM_IDs correspond to general heart failure readmissions of each patient.
general_readm = sorted_df.loc[~sorted_df['HADM_ID'].isin(most_recent_adm),'HADM_ID']
display(len(general_readm.unique()))

#Move the admission time upward by one place for each patient
sorted_df['shifted_admittime'] = sorted_df.groupby('SUBJECT_ID')['ADMITTIME'].shift(-1)
#Compute the interval between two successive admissions for each patient
sorted_df['interval'] = sorted_df['shifted_admittime'] - sorted_df['DISCHTIME']
#Isolate the 30-day heart failure readmissions
readm_30d = sorted_df.loc[sorted_df['interval'] <= pd.Timedelta(days=31), 'HADM_ID']
display(len(readm_30d))

14040

3604

969

In [3]:
#Read the NOTEEVENTS table and select the items that are Discharge summary
note_info = pd.read_csv('NOTEEVENTS.csv.gz', usecols=['SUBJECT_ID','HADM_ID','CATEGORY','TEXT'],compression='gzip')
discharge_sum=note_info[note_info['CATEGORY']=='Discharge summary']
#Isolate the Discharge summaries that belong to heart failure admissions
discharge_hf = discharge_sum[discharge_sum['HADM_ID'].isin(heart_failure_HADM)]
#Isolate the admissions that are related to heart failure and have at least a discharge summary
hf_adm_discharge = discharge_hf['HADM_ID'].unique()
display(len(hf_adm_discharge))

gen_readm_discharge = set(general_readm).intersection(set(hf_adm_discharge))
display(len(gen_readm_discharge))

readm_30d_discharge = set(readm_30d).intersection(set(hf_adm_discharge))
display(len(readm_30d_discharge))

13755

3544

963

In [4]:
import nltk
import re
from nltk.corpus import stopwords

#Download stopwords and define the stop words set
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

#Function to clean text: remove stop words and numbers
def clean_text(text):
    # Remove numbers using regex
    text = re.sub(r'\d+', '', text)
    # Split text into words
    words = text.split()
    # Remove stopwords
    words = [word for word in words if word.lower() not in stop_words]
    # Join words back into a cleaned text string
    cleaned_text = ' '.join(words)
    return cleaned_text

# Apply the cleaning function to each row in the 'TEXT' column
discharge_hf['cleaned_text'] = discharge_hf['TEXT'].apply(clean_text)

def text_length(text):
    return len(text.split())  # Count words by splitting by space

#Apply the text_length function to the 'cleaned_text' column
discharge_hf['text_length'] = discharge_hf['cleaned_text'].apply(text_length)

# Group by 'HADM_ID' and find the index of the row with the maximum 'text_length' for each group
idx = discharge_hf.groupby('HADM_ID')['text_length'].idxmax()

# Use .loc to select the rows based on the indices
HADM_len = discharge_hf.loc[idx, ['HADM_ID', 'cleaned_text']]
HADM_len['gen_readm']='negative'
HADM_len.loc[HADM_len['HADM_ID'].isin(gen_readm_discharge),'gen_readm']='positive'
HADM_len['readm_30d']='negative'
HADM_len.loc[HADM_len['HADM_ID'].isin(readm_30d_discharge),'readm_30d']='positive'
HADM_len.to_csv('HADM_readmin.csv', index=False)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
<ipython-input-4-dcf2e71e6047>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  discharge_hf['cleaned_text'] = discharge_hf['TEXT'].apply(clean_text)
<ipython-input-4-dcf2e71e6047>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  discharge_hf['text_length'] = discharge_hf['cleaned_text'].apply(text_length)


In [5]:
import pandas as pd

#Read the csv file and generate gen_readm_df and readm_30d_df
readm = pd.read_csv('HADM_readmin.csv')
gen_readm_df = readm[['HADM_ID', 'cleaned_text', 'gen_readm']]
readm_30d_df = readm[['HADM_ID', 'cleaned_text', 'readm_30d']]

#Under-sampling for gen_readm
gen_positive = gen_readm_df[gen_readm_df['gen_readm'] == 'positive']  # Select positives
gen_negative = gen_readm_df[gen_readm_df['gen_readm'] == 'negative']  # Select negatives

#Randomly sample negative cases to match the number of positives
gen_negative_sampled = gen_negative.sample(n=len(gen_positive), random_state=903965310)

#Concatenate positives and sampled negatives
gen_balanced = pd.concat([gen_positive, gen_negative_sampled])

#Under-sampling for readm_30d
readm_positive = readm_30d_df[readm_30d_df['readm_30d'] == 'positive']  # Select positives
readm_negative = readm_30d_df[readm_30d_df['readm_30d'] == 'negative']  # Select negatives

#Randomly sample negative cases to match the number of positives
readm_negative_sampled = readm_negative.sample(n=len(readm_positive), random_state=903965310)

#Concatenate positives and sampled negatives
readm_balanced = pd.concat([readm_positive, readm_negative_sampled])

#Now both gen_balanced and readm_balanced have equal number of positive and negative samples
display(gen_balanced['gen_readm'].value_counts(), readm_balanced['readm_30d'].value_counts())

,count
gen_readm,
positive,3544
negative,3544


,count
readm_30d,
positive,963
negative,963


In [6]:
pd.set_option('display.max_colwidth', None)

print('Admissions followed by readmissions\n')
print(gen_positive.shape)
display(gen_positive.head(1))

print('--------------------------\n\n')

print('Admissions followed by 30-d readmissions\n')
print(readm_positive.shape)
display(readm_positive.head(1))

Admissions followed by readmissions

(3544, 3)


HADM_ID  \
0  100018.0   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

--------------------------


Admissions followed by 30-d readmissions

(963, 3)


HADM_ID  \
1  100020.0   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     